<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Tiled copy and swizzle

`04_layout` built layouts and cut them into tiles. This one spreads a tile across
threads — the **thread-value layout** and **tiled copy** — partitions it, applies the
**swizzle** trick that keeps LDS banks happy, and *draws* all of it with `print_typst`.

As before, the layout algebra runs inside a `@flyc.jit` for the MLIR context; only
the copy capstone actually launches on the GPU.

In [ ]:
import os
import tempfile

# print_typst writes its .typ files as a trace-time side effect of @flyc.jit, and the
# layout cells print at trace time too. A warm JIT disk cache would skip the re-trace
# (and those side effects), so disable it — re-runs then always reproduce. Set before
# importing flydsl.
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"

import torch
from IPython.display import SVG, Code, display

import flydsl.compiler as flyc
import flydsl.expr as fx

try:
    import typst  # `pip install typst` — bundles the compiler
except ImportError:
    typst = None  # without it, diagrams fall back to raw source

_TYP_DIR = tempfile.mkdtemp()


def render_typst(name):
    """Show a Typst file written by print_typst: as an inline SVG when the `typst`
    package is available, otherwise as the raw Typst source."""
    path = os.path.join(_TYP_DIR, name)
    if typst is None:
        with open(path) as f:
            display(Code(f.read(), language="typst"))
        return
    out = typst.compile(path, format="svg")
    if isinstance(out, list):  # one entry per page; our files have a single page
        out = out[0]
    if isinstance(out, (bytes, bytearray)):
        out = out.decode()
    display(SVG(out))

## 1. Thread-value layouts

A copy hands a tile to a *team* of threads. Two layouts describe the split: a
**thread layout** — the shape of the thread grid over the tile — and a **value
layout** — how many elements each thread carries, and in what shape. `make_layout_tv`
folds them into one **TV layout** (thread, value → offset in the tile) plus the tile
shape it covers, and `make_tiled_copy` binds that to a copy atom.

Below, `thr` shape `(4, 1)` is 4 threads down dimension 0 (a 4×1 column); `val` shape
`(1, 8)` is 8 elements per thread along dimension 1 (a 1×8 row). Together they tile a
`(4, 8)` region — 4 threads × 8 elements — which is the `tile (M, N)` the cell prints.

In [ ]:
@flyc.jit
def tv_layout():
    thr = fx.make_layout((4, 1), (1, 1))  # 4 threads, laid out down the rows
    val = fx.make_layout((1, 8), (1, 1))  # each thread owns 8 columns
    tile_mn, tv = fx.make_layout_tv(thr, val)
    print("tile (M, N):", tile_mn)  # the (4, 8) region one copy covers
    print("TV layout  :", tv)  # (thread, value) -> offset in the tile

    atom = fx.make_copy_atom(fx.UniversalCopy128b(), fx.Float32)
    tiled = fx.make_tiled_copy(atom, tv, tile_mn)
    print("tiled copy :", tiled)


tv_layout()

Read the printed `TV layout (4,8):(1,4)` as shape `(4, 8)` = (4 threads, 8 values each);
the stride then places `(thread t, value v)` in the tile — step `1` per thread, `4` per
value (one value-step hops past all 4 threads).

## 2. Partitioning, and a copy that runs

`tiled.get_slice(tid)` gives one thread's view of the tiled copy; `partition_S` and
`partition_D` then carve the source and destination tiles into that thread's slice.
Each partition is a **memref view** — the same memory, but with a layout that maps only
the elements this thread copies (the per-thread coordinate bookkeeping is the
coord-tensor machinery from `04_layout`, which `partition_S/D` build for you). The
kernel below copies an `M×N` matrix tile-by-tile and checks the result against the
input; it also prints one thread's partitioned view at trace time.

The device path here is AMD CDNA-specific — `rocdl.make_buffer_tensor` +
`rocdl.BufferCopy128b`, the same atoms as `examples/02-tiledCopy.py`. The layout
pieces (`make_layout_tv`, `make_tiled_copy`, `partition_S/D`) are arch-neutral; only
the copy atom and the buffer wrapper are CDNA.

In [ ]:
@flyc.kernel
def tiled_copy_kernel(A: fx.Tensor, B: fx.Tensor):
    tid = fx.thread_idx.x
    bid = fx.block_idx.x
    A = fx.rocdl.make_buffer_tensor(A)
    B = fx.rocdl.make_buffer_tensor(B)
    bA = fx.slice(fx.zipped_divide(A, (8, 24)), (None, bid))
    bB = fx.slice(fx.zipped_divide(B, (8, 24)), (None, bid))

    thr = fx.make_layout((4, 1), (1, 1))
    val = fx.make_layout((1, 8), (1, 1))
    atom = fx.make_copy_atom(fx.rocdl.BufferCopy128b(), fx.Float32)
    tile_mn, tv = fx.make_layout_tv(thr, val)
    tiled = fx.make_tiled_copy(atom, tv, tile_mn)

    thr_copy = tiled.get_slice(tid)
    psrc = thr_copy.partition_S(bA)
    pdst = thr_copy.partition_D(bB)
    print("  per-thread source view:", psrc)  # trace-time: prints once, during compile

    frag = fx.make_fragment_like(psrc)
    fx.copy(atom, psrc, frag)
    fx.copy(atom, frag, pdst)


@flyc.jit
def run_copy(A: fx.Tensor, B: fx.Tensor, stream: fx.Stream = fx.Stream(None)):
    tiled_copy_kernel(A, B).launch(grid=(15, 1, 1), block=(4, 1, 1), stream=stream)


M, N = 8 * 3, 24 * 5
A = torch.arange(M * N, dtype=torch.float32).reshape(M, N).cuda()
B = torch.zeros(M, N, dtype=torch.float32).cuda()
run_copy(A, B, stream=torch.cuda.Stream())
torch.cuda.synchronize()
print("copy correct:", bool(torch.allclose(A, B)))

`copy correct: True` confirms the round-trip. The `per-thread source view` line printed
something like `Tensor<f32, buffer_desc, ((4,2),2,3):(…)>` — don't decode the nested
shape; the takeaway is just that it is *this thread's* slice of the same `f32` buffer
(`buffer_desc`), a view rather than freshly allocated memory.

## 3. Swizzle — keeping LDS banks happy

When threads stage a tile through shared memory (LDS, on AMD CDNA), a plain row-major
layout makes many of them hit the *same* bank on the same cycle — a bank conflict that
serializes the access. A **swizzle** scrambles the address with a cheap XOR so those
accesses spread across banks instead. `fx.SwizzleType.get(mask, base, shift)` describes
the scramble — the three numbers tune which address bits get XORed and by how much, knobs
you copy from a known-good config (the GEMM kernels) rather than derive by hand — and
`make_composed_layout` stacks it on a base layout: a coordinate goes through the base
layout to an address, then the swizzle perturbs that address.

(There is a coordinate-space cousin, `coord_swizzle` — the same XOR applied to
coordinates rather than byte addresses. The layout builder reaches for it internally
when it assembles a swizzled layout, so it stays an implementation detail you never
construct directly.)

In [ ]:
@flyc.jit
def swizzle():
    base = fx.make_ordered_layout((8, 8), (1, 0))
    swz = fx.make_composed_layout(fx.static(fx.SwizzleType.get(3, 0, 3)), base)
    print("base layout   :", base)
    print("swizzle       :", fx.SwizzleType.get(3, 0, 3))
    print("composed (swz):", swz)


swizzle()

## 4. Seeing it — `print_typst`

Layouts, TV layouts, and swizzles are far easier to *see* than to read off a stride
tuple. `fx.utils.print_typst` writes a [Typst](https://typst.app) diagram of a layout,
a `TiledCopy`, or a composed swizzle; with `pip install typst` we compile that to SVG
and show it inline (without the package, the cell falls back to the raw Typst source).
The `print_typst` calls run inside the `@flyc.jit` (they need the layout's context);
the rendering happens afterwards on the host.

In [ ]:
@flyc.jit
def emit_diagrams():
    plain = fx.make_ordered_layout((8, 8), (1, 0))
    swz = fx.make_composed_layout(fx.static(fx.SwizzleType.get(3, 0, 3)), fx.make_ordered_layout((8, 8), (1, 0)))
    thr = fx.make_layout((4, 1), (1, 1))
    val = fx.make_layout((1, 8), (1, 1))
    tile_mn, tv = fx.make_layout_tv(thr, val)
    tiled = fx.make_tiled_copy(fx.make_copy_atom(fx.UniversalCopy128b(), fx.Float32), tv, tile_mn)

    fx.utils.print_typst(plain, file=os.path.join(_TYP_DIR, "plain.typ"))
    fx.utils.print_typst(swz, file=os.path.join(_TYP_DIR, "swizzle.typ"))
    fx.utils.print_typst(tiled, file=os.path.join(_TYP_DIR, "tiled_copy.typ"))


emit_diagrams()

An `8×8` layout, each cell shaded by its linear index — plain, then with the
`(3, 0, 3)` swizzle applied. The reshuffled shading in the second grid is the
bank-conflict fix made visible: cells that used to share an address bit now sit apart.

In [ ]:
render_typst("plain.typ")

In [ ]:
render_typst("swizzle.typ")

And the tiled copy from §1 — cells sharing a colour belong to the same thread, so each
colour block is one thread's 8 elements; you can read all four threads' slices straight
off the grid:

In [ ]:
render_typst("tiled_copy.typ")

That is the layout toolkit end to end: shapes and strides (`04`), tiling and the TV
layout, partitioning into per-thread views, the swizzle that keeps LDS fast, and a way
to see all of it. It is the same machinery behind `examples/02-tiledCopy.py` and the
preshuffle GEMM.

---
*End of the layout series for now. The MMA atoms — `make_mma_atom`, `make_tiled_mma`,
and the `gemm` op in `examples/03-tiledMma.py` — build directly on these pieces.*